In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [3]:
from ultralytics import YOLO
import os

# Class model class id to class colour
id_to_colour = {
    0: 'black', 1: 'blue', 2: 'brown', 3: 'gold', 4: 'gray',
    5: 'green', 6: 'orange', 7: 'red', 8: 'silver', 9: 'violet',
    10: 'white', 11: 'yellow'
}


# class colour to accurate id
colour_to_value = {
    'black': 0, 'brown': 1, 'red': 2, 'orange': 3, 'yellow': 4,
    'green': 5, 'blue': 6, 'violet': 7, 'gray': 8, 'white': 9,
    'gold': -1, 'silver': -2
}

# class colour to multiplier
multiplier  = {
    'black': 1, 'brown': 10, 'red': 100, 'orange': 1, 'yellow': 10,
    'green': 100, 'blue': 1, 'violet': 10, 'gray': 100, 'white': 1,
    'gold': 0.1, 'silver': 0.01
}

# class colour to tolerance
tolerance = {
    'brown': 1, 'red': 2, 'orange': 2, 'yellow': 4, 'green': 0.5,
    'blue': 0.25, 'violet': 0.1, 'gray': 0.05, 'gold': 5, 'silver': 10
}


# Load models
model = YOLO("/content/drive/MyDrive/Colab_Notebooks/Bands_model/runs/detect/train/weights/best.pt") # band detection model
model2 = YOLO("/content/drive/MyDrive/Colab_Notebooks/Resistor_model/runs/detect/train/weights/best.pt")  # resistor detection model

# Get image paths from folder
image_paths = []
folder_path = "/content/drive/MyDrive/Colab_Notebooks/Bands_model/custom_data/test/images"
for filename in os.listdir(folder_path):
    full_path = os.path.join(folder_path, filename)
    if os.path.isfile(full_path):
        image_paths.append(full_path)


# Run models
print(f"\n\nBand detection")
results = model(image_paths)
print(f"\n\nResistor detection")
results2 = model2(image_paths)


# Process each image
for idx, result in enumerate(results):
    result2 = results2[idx]
    print(f"\n Image {idx + 1}")

    detected_numbers = []

    boxes = result.boxes
    boxes2 = result2.boxes

    if boxes is not None and len(boxes) > 0:
        yx_condition_met = False    # Check the condition from the model2
        if boxes2 is not None and len(boxes2) > 0:  # if model2 detects y>x
            second_box = boxes2[0]
            x1, y1, x2, y2 = second_box.xyxy[0].tolist()
            x = x2-x1
            y = y2-y1
            if y > x:  # If y < x
                yx_condition_met = True

        if yx_condition_met:
          #image is verticle sort band detection top to bottom
            sorted_boxes = sorted(boxes, key=lambda box: box.xyxy[0][1].item())
            if int(sorted_boxes[0].cls[0]) in [3, 8] or int(sorted_boxes[1].cls[0]) in [3, 8]: # if gold or silver model ID dectected on first 2 boxes, reverse sorting
              sorted_boxes.reverse()
            if int(sorted_boxes[-1].cls[0]) in [0, 10]: # if black or white model ID dectected on last box, reverse sorting
              sorted_boxes.reverse()
        else:
          #image is horizontal sort band detection top to bottom
            sorted_boxes = sorted(boxes, key=lambda box: box.xyxy[0][0].item())
            if int(sorted_boxes[0].cls[0]) in [3, 8] or int(sorted_boxes[1].cls[0]) in [3, 8]:
              sorted_boxes.reverse()
            if int(sorted_boxes[-1].cls[0]) in [0, 10]:
              sorted_boxes.reverse()

        #calculating colour code, model ID to colour code
        for i, box in enumerate(sorted_boxes):
            cls_id = int(box.cls[0].item())
            colour_label = id_to_colour.get(cls_id, 'unknown')


            if i == len(sorted_boxes) - 2 and colour_label in multiplier :  #calculating colour to multiplyer,
                mul_val = multiplier [colour_label]
                if int(sorted_boxes[-2].cls[0]) in [0, 2, 7, 3, 8]:
                  detected_numbers.append(f", {mul_val}")
                  print(f"Colour: {colour_label} = multiplier : {mul_val}")
                elif int(sorted_boxes[-2].cls[0]) in [6, 11, 5]:    # multiplier are in the kilohm (K)
                  detected_numbers.append(f", {mul_val}K")
                  print(f"Coluor: {colour_label} = multiplier : {mul_val}K")
                elif int(sorted_boxes[-2].cls[0]) in [1, 9, 4]:   # multiplier are in the megaohms (M)
                  detected_numbers.append(f", {mul_val}M")
                  print(f"Colour: {colour_label} = multiplier : {mul_val}M")
                elif int(sorted_boxes[-2].cls[0]) in [10]:    # multiplier are in the gigaohms (G)
                  detected_numbers.append(f", {mul_val}G")
                  print(f"Colour: {colour_label} = multiplier : {mul_val}G")

            elif i == len(sorted_boxes) - 1 and colour_label in tolerance: #calculating colour to tolerance
                tol_val = tolerance[colour_label]
                detected_numbers.append(f", %{tol_val}")
                print(f"Colour: {colour_label} = tolerance: %{tol_val}")

            elif colour_label in colour_to_value:   #calculating colour to resistance
                value = colour_to_value[colour_label]
                detected_numbers.append(value)
                print(f"Colour: {colour_label} = Value: {value}")

        # Print final resistance
        result2.show()
        result.show()
        final_number = ''.join(map(str, detected_numbers))
        print(f"\nResistence: {final_number}")
    else:
        print("No boxes detected.")


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
print(model.names) # to check models colour ID

{0: 'black', 1: 'blue', 2: 'brown', 3: 'gold', 4: 'gray', 5: 'green', 6: 'orange', 7: 'red', 8: 'silver', 9: 'violet', 10: 'white', 11: 'yellow'}
